# Decoder Inference (Letter-based AUROC)

This notebook evaluates decoder LLMs on glioma prognostication tasks using letter-based prompts (A/B/C) and calibrated AUROC from token logprobs. Secrets (API keys/endpoints) are taken from environment variables and are not stored in the notebook.


In [ ]:
import os
import logging
from typing import List, Dict

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report, confusion_matrix

try:
    from openai import OpenAI, AzureOpenAI
except Exception:
    raise RuntimeError("Please install the OpenAI client library in this environment.")

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

# Load secrets from env (never hard-code!)
MODEL_CHOICE = os.environ.get('DECODER_MODEL', 'biomistral')  # 'biomistral' or 'gpt-4o'
VLLM_BASE_URL = os.environ.get('VLLM_BASE_URL', 'http://localhost:8000/v1')
AZURE_API_KEY = os.environ.get('AZURE_OPENAI_API_KEY', '')
AZURE_API_VERSION = os.environ.get('AZURE_OPENAI_API_VERSION', '2024-10-21')
AZURE_ENDPOINT = os.environ.get('AZURE_OPENAI_ENDPOINT', '')


In [ ]:
def setup_client(model_choice: str):
    if model_choice.lower() == 'biomistral':
        client = OpenAI(base_url=VLLM_BASE_URL, api_key=os.environ.get('VLLM_API_KEY', 'token-abc123'))
        model_name = 'BioMistral-7B'
        logger.info(f"Using local vLLM model: {model_name}")
        return client, model_name
    elif model_choice.lower() == 'gpt-4o':
        if not (AZURE_API_KEY and AZURE_ENDPOINT):
            raise RuntimeError('Set AZURE_OPENAI_API_KEY and AZURE_OPENAI_ENDPOINT in the environment.')
        client = AzureOpenAI(api_key=AZURE_API_KEY, api_version=AZURE_API_VERSION, azure_endpoint=AZURE_ENDPOINT)
        model_name = 'gpt-4o-2024-08-06'
        logger.info(f"Using Azure OpenAI model: {model_name}")
        return client, model_name
    else:
        raise ValueError("MODEL_CHOICE must be 'biomistral' or 'gpt-4o'")

client, MODEL_NAME = setup_client(MODEL_CHOICE)


In [ ]:
def get_task_labels(task: str) -> List[str]:
    if task == 'tumor_type':
        return ['glioblastoma', 'astrocytoma', 'oligodendroglioma']
    if task == 'survival':
        return ['NOT_SURVIVE', 'SURVIVE']
    if task == 'mgmt':
        return ['NEGATIVE', 'POSITIVE']
    raise ValueError(f'Unknown task: {task}')


def build_letter_prompt(clinical_note: str, task: str) -> str:
    labels = get_task_labels(task)
    options = '\n'.join([f"{chr(ord('A')+i)}: {lab}" for i, lab in enumerate(labels)])
    headers = {
        'tumor_type': 'You are an expert neuro-radiologist. Classify the brain tumor type.',
        'survival': 'You are an expert neuro-oncologist. Predict 1-year survival.',
        'mgmt': 'You are an expert molecular pathologist. Predict MGMT promoter methylation.'
    }
    header = headers[task]
    return f"""{header}

Clinical Notes:
{clinical_note}

Choose from:
{options}

Answer with a single letter:"""


In [ ]:
def predict_letter_with_probs(client, model_name: str, clinical_note: str, task: str):
    import numpy as np
    labels = get_task_labels(task)
    prompt = build_letter_prompt(clinical_note, task)
    letter_to_idx = {chr(ord('A')+i): i for i in range(len(labels))}

    completion = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=5,
        logprobs=True,
        top_logprobs=10,
    )

    # Parse letter
    response = (completion.choices[0].message.content or '').strip().upper()
    pred_idx = -1
    for ch in response:
        if ch in letter_to_idx:
            pred_idx = letter_to_idx[ch]
            break

    # Default uniform
    probs = np.ones(len(labels), dtype=float) / len(labels)

    # Extract token logprobs for first token
    lp = getattr(completion.choices[0], 'logprobs', None)
    if lp and getattr(lp, 'content', None):
        top = lp.content[0].top_logprobs
        logprob_map = {ltr: -100.0 for ltr in letter_to_idx}
        for entry in top:
            tok = (entry.token or '').upper()
            if len(tok) == 1 and tok in letter_to_idx:
                logprob_map[tok] = entry.logprob
        logvals = np.array([logprob_map[ltr] for ltr in sorted(letter_to_idx.keys())])
        expv = np.exp(logvals - np.max(logvals))
        probs = expv / expv.sum()

    return pred_idx, probs


In [ ]:
def evaluate_task(client, model_name: str, df: pd.DataFrame, label_col: str, task: str, text_col_candidates=('report','text','Anon Report Text'), n_splits=10, batch_size=30):
    # Pick text column
    text_col = next((c for c in text_col_candidates if c in df.columns), None)
    if text_col is None:
        raise ValueError('No valid text column found')

    # Filter valid labels
    valid_mask = df[label_col].notna()
    df = df[valid_mask]

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    fold_metrics = []
    y_true_all, y_pred_all = [], []

    for fold_idx, (_, test_idx) in enumerate(kf.split(df), 1):
        test_df = df.iloc[test_idx]
        notes = test_df[text_col].tolist()

        preds, probs = [], []
        for i in range(0, len(notes), batch_size):
            batch = notes[i:i+batch_size]
            for note in batch:
                pred_idx, pvec = predict_letter_with_probs(client, model_name, note, task)
                preds.append(pred_idx)
                probs.append(pvec)

        y_true = test_df[label_col].values.astype(int)
        y_pred = np.array(preds)
        y_prob = np.array(probs)

        # Metrics (filter invalid)
        valid = y_pred != -1
        if not np.any(valid):
            metrics = {"accuracy":0.0,"precision":0.0,"recall":0.0,"f1_macro":0.0,"aucroc":np.nan}
        else:
            yt, yp, yp_prob = y_true[valid], y_pred[valid], y_prob[valid]
            acc = accuracy_score(yt, yp)
            pr, rc, f1, _ = precision_recall_fscore_support(yt, yp, average='macro', zero_division=0)
            try:
                uniq = np.unique(yt)
                if len(uniq) == 2:
                    auc = roc_auc_score(yt, yp_prob[:,1])
                else:
                    auc = roc_auc_score(yt, yp_prob, multi_class='ovr')
            except Exception:
                auc = np.nan
            metrics = {"accuracy":acc,"precision":pr,"recall":rc,"f1_macro":f1,"aucroc":auc}

        fold_metrics.append(metrics)
        y_true_all.append(yt if np.any(valid) else np.array([]))
        y_pred_all.append(yp if np.any(valid) else np.array([]))

    # Summaries
    keys = ["accuracy","precision","recall","f1_macro","aucroc"]
    row = []
    for k in keys:
        vals = np.array([m.get(k, np.nan) for m in fold_metrics], dtype=float)
        row.append((np.nanmean(vals), np.nanstd(vals)))

    return fold_metrics, (np.concatenate(y_true_all) if y_true_all else np.array([])), (np.concatenate(y_pred_all) if y_pred_all else np.array([]))


In [ ]:
# Load data
TUMOR_PATH = os.environ.get('TUMOR_EVAL_CSV', 'data/eval/tumor_type_eval.csv')
MGMT_PATH = os.environ.get('MGMT_EVAL_CSV', 'data/eval/mgmt_eval.csv')
OS_PATH = os.environ.get('OS_EVAL_CSV', 'data/eval/os_eval.csv')

tumor_df = pd.read_csv(TUMOR_PATH)
mgmt_df = pd.read_csv(MGMT_PATH)
os_df = pd.read_csv(OS_PATH)
mgmt_df['mgmt_label'] = ((mgmt_df['mgmt'].notna()) & (mgmt_df['mgmt'] == 'positive')).astype(int)

# Filter to 3-class tumor task
if 'tumor_type' in tumor_df.columns:
    tumor_df = tumor_df[tumor_df['tumor_type'].str.lower().isin(get_task_labels('tumor_type'))].copy()


In [ ]:
# Evaluate tasks
folds, y_true, y_pred = evaluate_task(client, MODEL_NAME, tumor_df, 'class_label', 'tumor_type', n_splits=10, batch_size=30)

# Summaries
metrics_order = ['accuracy','precision','recall','f1_macro','aucroc']
row = [MODEL_NAME]
for k in metrics_order:
    vals = np.array([m.get(k, np.nan) for m in folds], dtype=float)
    row.append(f"{np.nanmean(vals)*100:.1f} ± {np.nanstd(vals)*100:.1f}")

print('|Model|Accuracy (%)|Precision (%)|Recall (%)|F1-macro (%)|AUCROC (%)|')
print('|---|---|---|---|---|---|')
print('|'+'|'.join(row)+'|')

# Classification report and confusion matrix
if y_true.size and y_pred.size:
    names = [ 'glioblastoma','astrocytoma','oligodendroglioma' ]
    print('\n'+classification_report(y_true, y_pred, target_names=names, digits=3))
    cm = confusion_matrix(y_true, y_pred, labels=sorted(np.unique(y_true)))
    display(pd.DataFrame(cm, index=names, columns=names))
